# CPSA 03 — FP32 training on 2× T4

YOLOv8-pose with Hard-Swish, no attention, trained on the **mixed**
dataset: the half-bust crops from notebook 01 plus the full-frame images
from notebook 02, so the predicted box learns the true person extent while
half-body views stay dominant. Ultralytics trains square `imgsz=640` with
`rect=False`; the deployment graph is exported and audited at the camera's
native 480×640 input.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run.
COMMIT_SHA = "c5f64644f5080f4bfa593159800385826d60233d"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from cpsa_ml.train import TrainConfig, train
from cpsa_ml.variant_study import build_mixed_data_yaml

# Kaggle mounts datasets under /kaggle/input/datasets/<owner>/<slug>/ (older
# kernels: /kaggle/input/<slug>/), so datasets are located by content.
INPUT = Path("/kaggle/input")
DATA = next((p for p in sorted(INPUT.rglob("data.yaml")) if "cpsa-halfbust" in str(p)), None)
if DATA is None:
    raise FileNotFoundError("attach fabiocabeccia/cpsa-halfbust (no data.yaml found)")
FULL_DATA = next(
    (p.parent for p in sorted(INPUT.rglob("data.yaml")) if "cpsa-fullframe" in str(p)), None
)  # notebook-02 dataset; None trains on the half-body set only
print("half-body data:", DATA, "| full-frame data:", FULL_DATA)
MIXED_DATA = Path("/kaggle/working/data_mixed.yaml")
SCHEMA = "halfbust62"
VARIANT = "n"
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"  # 2× T4 DDP
IMGSZ = 640
PROJECT = Path("/kaggle/working/runs")
NAME = "halfbust62-yolov8n-hswish-fp32"
RESUME = ""  # e.g. "/kaggle/input/cpsa-checkpoints/last.pt"

payload = build_mixed_data_yaml(DATA.parent, FULL_DATA, MIXED_DATA, schema=SCHEMA)
print("train:", payload["train"])
print("val_full:", payload.get("val_full"))

best = train(
    TrainConfig(
        data=MIXED_DATA,
        schema=SCHEMA,
        variant=VARIANT,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        imgsz=IMGSZ,
        project=PROJECT,
        name=NAME,
        resume=Path(RESUME) if RESUME else None,
        seed=0,
        workers=4,
        amp=True,
        val=True,
        plots=True,
    )
)
print("best checkpoint:", best)

In [ ]:
from cpsa_ml.onnx_check import check_onnx, export_onnx, format_report

ONNX = Path("/kaggle/working/export/halfbust62.onnx")
onnx_path = export_onnx(best, ONNX, imgsz=(480, 640))
report = check_onnx(onnx_path, imgsz=(480, 640), opset=17)
print(format_report(report))
assert report.passed, "backbone/neck is not DPU-compatible"

In [ ]:
# Re-publish /kaggle/working/runs as a dataset version. In the next session set
# RESUME to the attached .../weights/last.pt.
for path in sorted((PROJECT / NAME / "weights").glob("*.pt")):
    print(path)